# L1-Regularized Logistic Regression Baseline

Untreated MS patients + their household controls (418 samples), using the 93 shared genus with the Levi cohort.

- Zhou values at or below Levi's 0.0001 detection floor are set to 0, so both cohorts are measured the same way
- each sample is renormalized over the 93 genus
- splits are made by **household**, so paired MS + Control never land on opposite sides

In [8]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import MinMaxScaler

import warnings

warnings.filterwarnings('ignore', category = FutureWarning)
warnings.filterwarnings('ignore', message = 'Inconsistent values')


In [9]:
def build_training_set(zhou_path = '/Users/noe/Coding/MSSE/277B/gut-microbiome-signature-multiple-sclerosis/data/preprocessed/merged_dfs/zhou_genus_merged_df.csv',
                       levi_path = '/Users/noe/Coding/MSSE/277B/gut-microbiome-signature-multiple-sclerosis/data/preprocessed/abundance_data/Cohort2_Levi_Genus.csv',
                       detection_floor = 0.0001):
    
    merged = pd.read_csv(zhou_path, low_memory = False)

    # untreated MS patients plus the control from each of their households
    untreated = merged[merged['treatment_status'] == 'Untreated']
    housemates = merged[(merged['household'].isin(untreated['household'])) & (merged['disease'] == 'Control')]
    training_phenotype = pd.concat([untreated, housemates]).reset_index(drop = True)

    # the shared genus
    zhou_genus = merged.columns[merged.columns.get_loc('Cohort') + 1:]
    levi_genus = pd.read_csv(levi_path, nrows = 1).columns[1:]
    shared_genus = sorted(set(zhou_genus) & set(levi_genus))

    # zero anything Levi couldn't have detected, then rescale each sample to sum to 1 over the 93
    abundance = training_phenotype[shared_genus]
    abundance = abundance.where(abundance > detection_floor, 0.0)
    abundance = abundance.div(abundance.sum(axis = 1), axis = 0)
    return training_phenotype, abundance, shared_genus


training_phenotype, abundance, shared_genus = build_training_set()

X = abundance.to_numpy()
y = (training_phenotype['disease'] == 'MS').astype(int).to_numpy()
sex_rule = (training_phenotype['sex'] == 'F').astype(int).to_numpy()
households = training_phenotype['household']

print(f'Training set: {X.shape[0]} samples x {X.shape[1]} genus, MS = {y.sum()}')
print(f'Samples left empty by the floor: {(abundance.sum(axis = 1) == 0).sum()}')

Training set: 418 samples x 93 genus, MS = 209
Samples left empty by the floor: 0


### Split by Household

In [10]:
# split the households so ms+control household pairing are always on the same side
household_ids = households.unique()
train_households, test_households = train_test_split(household_ids, test_size = 0.2, random_state = 42)
is_train = households.isin(train_households).to_numpy()

X_train, y_train, sex_train = X[is_train], y[is_train], sex_rule[is_train]
X_test, y_test, sex_test = X[~is_train], y[~is_train], sex_rule[~is_train]
train_household_labels = households[is_train].to_numpy()

print(f'Train: {is_train.sum()} samples ({len(train_households)} households)')
print(f'Test:  {(~is_train).sum()} samples ({len(test_households)} households)')

Train: 334 samples (167 households)
Test:  84 samples (42 households)


### 5-Fold CV Over Training Households to Choose C

In [11]:
C_values = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0]
kfold = KFold(n_splits = 5, shuffle = True, random_state = 42)
fold_splits = list(kfold.split(train_households))

cv_results = []
fold_coefficients = {}

for C in C_values:

    fold_accuracies = []
    fold_sex_accuracies = []
    coefficients = []

    for fold_train_idx, fold_val_idx in fold_splits:

        # map this fold's training households back to their samples
        is_fold_train = np.isin(train_household_labels, train_households[fold_train_idx])

        # fit the scaler on the fold's training part only
        scaler = MinMaxScaler()
        X_fold_train = scaler.fit_transform(X_train[is_fold_train])
        X_fold_val = scaler.transform(X_train[~is_fold_train])

        model = LogisticRegression(penalty = 'l1', solver = 'liblinear', C = C)
        model.fit(X_fold_train, y_train[is_fold_train])
        y_pred = model.predict(X_fold_val)

        fold_accuracies.append((y_pred == y_train[~is_fold_train]).sum() / len(y_pred))
        fold_sex_accuracies.append((sex_train[~is_fold_train] == y_train[~is_fold_train]).mean())
        coefficients.append(model.coef_[0])

    # keep each fold's coefficients for the table
    coefficients = np.array(coefficients)
    fold_coefficients[C] = coefficients

    cv_results.append({'C': C,
                       'cv_accuracy': np.mean(fold_accuracies),
                       'cv_std': np.std(fold_accuracies),
                       'sex_rule_cv': np.mean(fold_sex_accuracies),
                       'mean_genus_kept': (coefficients != 0).sum(axis = 1).mean(),
                       'genus_ever_kept': (coefficients != 0).any(axis = 0).sum(),
                       'genus_kept_all_5': (coefficients != 0).all(axis = 0).sum()})

cv_table = pd.DataFrame(cv_results)
cv_table.round(3)

,C,cv_accuracy,cv_std,sex_rule_cv,mean_genus_kept,genus_ever_kept,genus_kept_all_5
0,0.01,0.500,0.000,0.65,0.0,0,0
1,0.05,0.500,0.000,0.65,0.0,0,0
2,0.10,0.500,0.000,0.65,0.0,0,0
3,0.50,0.590,0.060,0.65,7.2,17,1
4,1.00,0.593,0.080,0.65,19.0,40,4
5,5.00,0.545,0.057,0.65,61.0,84,28


### Refit on the Full Training Set + Evaluation on the test set

In [12]:
best_C = cv_table.loc[cv_table['cv_accuracy'].idxmax(), 'C']

# refit on all training households with the chosen C
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

final_model = LogisticRegression(penalty = 'l1', solver = 'liblinear', C = best_C)
final_model.fit(X_train_scaled, y_train)

train_accuracy = (final_model.predict(X_train_scaled) == y_train).sum() / len(y_train)
test_accuracy = (final_model.predict(X_test_scaled) == y_test).sum() / len(y_test)

print(f'Chosen C = {best_C}')
print(f'Train accuracy: {train_accuracy:.3f}')
print(f'Test accuracy:  {test_accuracy:.3f}  (n = {len(y_test)})')
print(f'Sex rule on the same test set: {(sex_test == y_test).mean():.3f}')
print(f'Genus kept in final model: {(final_model.coef_[0] != 0).sum()} of {len(shared_genus)}')

Chosen C = 1.0
Train accuracy: 0.659
Test accuracy:  0.536  (n = 84)
Sex rule on the same test set: 0.726
Genus kept in final model: 19 of 93


### Stability: How Often Each Genus Is Kept Across the 5 Folds

In [13]:
best_coefficients = fold_coefficients[best_C]
stability = pd.DataFrame({'genus': shared_genus,
                          'folds_selected': (best_coefficients != 0).sum(axis = 0),
                          'mean_coefficient': best_coefficients.mean(axis = 0),
                          'final_coefficient': final_model.coef_[0]})

# keep genus chosen in at least 4 of 5 folds, strongest first
stable = stability[stability['folds_selected'] >= 4].copy()
stable['direction'] = np.where(stable['mean_coefficient'] > 0, 'higher in MS', 'higher in control')
stable = stable.reindex(stable['mean_coefficient'].abs().sort_values(ascending = False).index)

print(f'Genus selected in >= 4 of 5 folds at C = {best_C}: {len(stable)}')
stable.round(3)

Genus selected in >= 4 of 5 folds at C = 1.0: 11


,genus,folds_selected,mean_coefficient,final_coefficient,direction
41,faecalibacterium,5,-1.749,-1.962,higher in control
50,hungatella,5,1.033,1.542,higher in MS
78,prevotella,5,-0.923,-0.949,higher in control
80,roseburia,4,-0.551,-0.771,higher in control
43,flavonifractor,4,0.467,0.565,higher in MS
35,eisenbergiella,4,0.465,0.588,higher in MS
27,coprococcus,5,-0.464,-0.560,higher in control
16,bilophila,4,0.422,0.538,higher in MS
51,intestinibacter,4,-0.405,-0.479,higher in control
69,oscillibacter,4,0.290,0.349,higher in MS
